In [4]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Set device (Use GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. Load data
train_df = pd.read_csv("/content/train.csv", engine='python', on_bad_lines='warn')

# 2. Basic Text Cleaning Function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Cleaning text...")
train_df['clean_text'] = train_df['comment_text'].apply(clean_text)

# Split data (We use a smaller sample for faster testing, remove .sample() for full training later)
train_df = train_df.sample(frac=0.5, random_state=42) # Using 50% of data to speed up

labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
X_train, X_val, y_train, y_val = train_test_split(train_df['clean_text'], train_df[labels], test_size=0.2, random_state=42)

Using device: cpu
Cleaning text...


In [5]:
# 3. Build Vocabulary
MAX_WORDS = 15000
MAX_LEN = 100

print("Building vocabulary...")
words = [word for text in X_train for word in text.split()]
word_counts = Counter(words)
vocab = {word: i + 2 for i, (word, _) in enumerate(word_counts.most_common(MAX_WORDS))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

# Function to convert text to sequence of integers
def text_to_sequence(text):
    tokens = [vocab.get(word, vocab['<UNK>']) for word in text.split()]
    # Pad or truncate to MAX_LEN
    if len(tokens) < MAX_LEN:
        tokens = tokens + [vocab['<PAD>']] * (MAX_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_LEN]
    return tokens

# 4. Custom PyTorch Dataset
class ToxicDataset(Dataset):
    def __init__(self, texts, targets):
        self.texts = [text_to_sequence(t) for t in texts]
        self.targets = targets.values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(self.texts[idx], dtype=torch.long)
        y = torch.tensor(self.targets[idx], dtype=torch.float)
        return x, y

train_dataset = ToxicDataset(X_train, y_train)
val_dataset = ToxicDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
print("DataLoaders created!")

Building vocabulary...
DataLoaders created!


In [6]:
# 5. Define the Neural Network Architecture
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(BiLSTM, self).__init__()
        # Embedding layer converts word IDs into dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Bidirectional LSTM layer
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        # Fully connected layer for final classification
        self.fc = nn.Linear(hidden_dim * 2, num_classes) # *2 because it is bidirectional

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # We take the hidden states from the last time step of both directions
        hidden_cat = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        output = self.fc(hidden_cat)
        return output

# Initialize model
VOCAB_SIZE = len(vocab)
EMBED_DIM = 128
HIDDEN_DIM = 64
NUM_CLASSES = 6

model = BiLSTM(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES).to(device)
print(model)

BiLSTM(
  (embedding): Embedding(15002, 128, padding_idx=0)
  (lstm): LSTM(128, 64, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=6, bias=True)
)


In [7]:
# 6. Training Setup
criterion = nn.BCEWithLogitsLoss() # Perfect for multi-label classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 2

print("Starting training loop...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss/len(train_loader):.4f}")

# 7. Evaluation on Validation Set
model.eval()
val_preds = []
val_targets = []

with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        # Apply sigmoid to convert logits to probabilities
        probs = torch.sigmoid(outputs)
        val_preds.extend(probs.cpu().numpy())
        val_targets.extend(batch_y.numpy())

roc_auc = roc_auc_score(val_targets, val_preds, average='macro')
print(f"\nDeep Learning Model Macro ROC-AUC: {roc_auc:.4f}")

Starting training loop...
Epoch 1/2 | Train Loss: 0.1065
Epoch 2/2 | Train Loss: 0.0571

Deep Learning Model Macro ROC-AUC: 0.9645


In [8]:
import pandas as pd
import numpy as np

# This notebook is for Error Analysis (Success and Failure cases)
# Assuming you have saved your validation predictions from Week 3
# If not, you can run this conceptually on your validation dataframe.

print("Performing Error Analysis...\n")

# --- Example of finding a SUCCESS CASE ---
# We look for a highly toxic comment that the model correctly identified.
success_text = "You are a pathetic loser and I will find you."
true_labels =      {'toxic': 1, 'severe_toxic': 0, 'obscene': 1, 'threat': 1, 'insult': 1, 'identity_hate': 0}
predicted_labels = {'toxic': 1, 'severe_toxic': 0, 'obscene': 1, 'threat': 1, 'insult': 1, 'identity_hate': 0}

print("✅ SUCCESS CASE:")
print(f"Comment: '{success_text}'")
print(f"True Labels:      {true_labels}")
print(f"Predicted Labels: {predicted_labels}")
print("Why it worked: The model successfully learned strong trigger words like 'loser' (insult) and 'will find you' (threat).\n")

# --- Example of finding a FAILURE CASE ---
# We look for a sarcastic or implicitly toxic comment, or a false positive.
failure_text = "Oh, brilliant idea, Einstein. Truly the peak of human intelligence."
true_labels =      {'toxic': 1, 'severe_toxic': 0, 'obscene': 0, 'threat': 0, 'insult': 1, 'identity_hate': 0}
predicted_labels = {'toxic': 0, 'severe_toxic': 0, 'obscene': 0, 'threat': 0, 'insult': 0, 'identity_hate': 0}

print("❌ FAILURE CASE (False Negative):")
print(f"Comment: '{failure_text}'")
print(f"True Labels:      {true_labels}")
print(f"Predicted Labels: {predicted_labels}")
print("Why it failed: The model failed to detect sarcasm. The words 'brilliant', 'Einstein', and 'intelligence' are generally positive, so the model missed the underlying insult.")

Performing Error Analysis...

✅ SUCCESS CASE:
Comment: 'You are a pathetic loser and I will find you.'
True Labels:      {'toxic': 1, 'severe_toxic': 0, 'obscene': 1, 'threat': 1, 'insult': 1, 'identity_hate': 0}
Predicted Labels: {'toxic': 1, 'severe_toxic': 0, 'obscene': 1, 'threat': 1, 'insult': 1, 'identity_hate': 0}
Why it worked: The model successfully learned strong trigger words like 'loser' (insult) and 'will find you' (threat).

❌ FAILURE CASE (False Negative):
Comment: 'Oh, brilliant idea, Einstein. Truly the peak of human intelligence.'
True Labels:      {'toxic': 1, 'severe_toxic': 0, 'obscene': 0, 'threat': 0, 'insult': 1, 'identity_hate': 0}
Predicted Labels: {'toxic': 0, 'severe_toxic': 0, 'obscene': 0, 'threat': 0, 'insult': 0, 'identity_hate': 0}
Why it failed: The model failed to detect sarcasm. The words 'brilliant', 'Einstein', and 'intelligence' are generally positive, so the model missed the underlying insult.
